In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from ipywidgets import Dropdown, SelectMultiple, interactive_output, VBox, HBox, IntSlider, Layout, Label, HTML
from IPython.display import display, HTML as IPHTML
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [2]:
# load dataset dataset_1980_2020.csv
crime_df = pd.read_csv('dataset_1980_2020.csv')
crime_df.head()
crime_df = crime_df.rename(columns={'Count': 'Year'})

## Visualize Crime Statistics by State
Create a chorpleth map that can be filtered by the various crime statistics and year

In [3]:
def plot_crime_map(df, filters, year=None):
    # filter by year
    if year is not None:
        df = df[df['Year'] == year]

    valid_filters = [f for f in filters if f in df.columns]

    if not valid_filters:
        raise ValueError("no valid filters provided")
    
    # sum accross the selecter filter columns
    df['Filtered Crime Count'] = df[valid_filters].sum(axis=1)
    state_crime = df.groupby('State')['Filtered Crime Count'].sum().reset_index()

    # plot
    fig = px.choropleth(
        state_crime,
        locations='State',
        locationmode='USA-states',
        color='Filtered Crime Count',
        scope="usa",
        color_continuous_scale="Reds",
        labels={'Filtered Crime Count': 'Crime Count'},
        title=f"Crime Map - Filters: {', '.join(filters)}" + (f" ({year})" if year else "")
    )
    fig.show()

def interactive_crime_map(crime_df):
    all_filters = [
        '0 to 11', '12 to 17',
        'Male', 'Female', 'Unknown Gender',
        'White', 'Black', 'Amer. Indian/Alaskan Native', 'Asian/Nat. Hawaiian/Pac Isl', 'Unknown Race',
        'Family', 'Acquaintance', 'Stranger', 'Unknown',
        'Firearm', 'Knife', 'Blunt object', 'Personal', 'Other/unknown Weapon',
        'One offender involved', 'Two or more offenders involved', 'Unknown number of offenders involved'
    ]

    year_options = crime_df['Year'].dropna().unique()
    filter_label = HTML(
        value="<b>Select Filters</b>",
        layout=Layout(margin='0 0 10px 0')
    )  
    year_slider = IntSlider(
        value=int(year_options.min()),
        min=int(year_options.min()),
        max=int(year_options.max()),
        step=1,
        description='Year:',
        continuous_update=False,
        layout=Layout(width='100%', margin='20px 0 0 0')
    )
    year_slider.style = {
        'description_width': '80px',
        'handle_color': '#d62728',
        'font_size': '16px'
    }
    filter_select = SelectMultiple(
        options=all_filters,
        value=('Male',),
        rows=25,
        description = '',
        layout=Layout(
            width='100%',
            height='auto',
            overflow_y='visible',
            white_space='normal'
        )
    )
    filter_select.style = {'description_width': '0px'}


    left_panel = VBox([filter_label, filter_select], layout=Layout(width='300px', height='100%', align_items='stretch'))
    display(IPHTML("<style>.widget-readout { display: none !important; }</style>"))
    
    out = interactive_output(
        lambda filters, year: plot_crime_map(crime_df, list(filters), year),
        {'filters': filter_select, 'year': year_slider}
    )

    right_panel = VBox(
        [out, year_slider],
        layout=Layout(flex='1', height='100%', align_items='stretch')
    )
    full_ui = HBox(
        [left_panel, right_panel],
        layout=Layout(width='100%', height='auto', align_items='flex-start')
    )
    display(full_ui)

interactive_crime_map(crime_df)


## Forecast Future Crime Patterns and Assign Risk Scores ny State

In [4]:
# aggregate offender demographics by state-year
# offender demographics
offender_columns = [
    '0 to 11', '12 to 17',           # Age
    'Male', 'Female', 'Unknown Gender',  # Gender
    'White', 'Black', 'Amer. Indian/Alaskan Native',
    'Asian/Nat. Hawaiian/Pac Isl', 'Unknown Race'  # Race
]

# aggreate by state-year
crime_df = crime_df[crime_df['Year'] <= 2017]
agg = crime_df.groupby(['State', 'Year'])[offender_columns].sum().reset_index()

# normalize the feature matrix
features = agg[offender_columns]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
features_scaled_df = pd.DataFrame(features_scaled, columns=offender_columns)

# create final modeling df
model_df = pd.concat([agg[['State', 'Year']], features_scaled_df], axis=1)

model_df

,State,Year,0 to 11,12 to 17,Male,Female,Unknown Gender,White,Black,Amer. Indian/Alaskan Native,Asian/Nat. Hawaiian/Pac Isl,Unknown Race
0,AK,1980,-0.345967,-0.513364,-0.507367,-0.632547,-0.105104,-0.420300,-0.613506,2.625142,-0.226491,-0.174419
1,AK,1981,-0.345967,-0.492127,-0.485017,-0.632547,-0.105104,-0.302427,-0.613506,-0.336557,-0.226491,-0.174419
2,AK,1982,-0.345967,-0.364701,-0.395620,-0.014718,-0.105104,-0.184555,-0.613506,4.105991,-0.226491,-0.174419
3,AK,1983,1.438598,-0.513364,-0.485017,-0.632547,-0.105104,-0.381009,-0.613506,2.625142,-0.226491,-0.174419
4,AK,1984,-0.345967,-0.428414,-0.417969,-0.632547,-0.105104,-0.184555,-0.613506,-0.336557,-0.226491,-0.174419
...,...,...,...,...,...,...,...,...,...,...,...,...
1821,WY,2013,-0.345967,-0.513364,-0.507367,-0.632547,-0.105104,-0.420300,-0.613506,2.625142,-0.226491,-0.174419
1822,WY,2014,-0.345967,-0.534602,-0.529716,-0.632547,-0.105104,-0.420300,-0.613506,1.144293,-0.226491,-0.174419
1823,WY,2015,-0.345967,-0.555839,-0.552066,-0.632547,-0.105104,-0.420300,-0.613506,-0.336557,-0.226491,-0.174419
1824,WY,2016,-0.345967,-0.555839,-0.552066,-0.632547,-0.105104,-0.420300,-0.613506,-0.336557,-0.226491,-0.174419


In [5]:
# k means clustering
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(features_scaled)
model_df['Pattern'] = cluster_labels


# analyze cluster charactersistics 
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=offender_columns
)

centroids_original = pd.DataFrame(
    scaler.inverse_transform(centroids),
    columns=offender_columns
)

centroids_original.index.name = 'Pattern'

# save pattern assignments for time-series modeling and risk scoring
model_df.to_csv("state_year_pattern.csv", index=False)

centroids_original

,0 to 11,12 to 17,Male,Female,Unknown Gender,White,Black,Amer. Indian/Alaskan Native,Asian/Nat. Hawaiian/Pac Isl,Unknown Race
Pattern,,,,,,,,,,
0,0.034252,9.582278,9.072226,0.813850,0.008191,4.238273,5.145197,2.323157e-01,0.190618,0.073716
1,1.375000,182.703125,171.515625,12.531250,0.062500,91.562500,86.406250,5.156250e-01,4.390625,1.187500
2,1.875000,414.750000,399.000000,17.500000,0.125000,258.625000,129.750000,1.125000e+00,18.875000,8.625000
3,0.497525,47.650990,44.992574,4.074257,0.049505,14.257426,34.121287,1.509901e-01,0.324257,0.252475
4,0.571429,94.285714,82.142857,8.285714,4.428571,21.714286,62.142857,2.775558e-17,0.285714,11.000000


### Analysis of Patterns

#### Pattern 0
| *Feature* | *Notable Values* |
| --- | --- |
| Male | 9 |
| Black | 5 |
| White | 4 |
| 12 to 17 | 9 |

**Interpretation:**
- Low-volume demographic pattern
- Primarily male offenders, slightly more Black than White
- Young age group (12–17) slightly more prevalent
- Possibly small, localized juvenile crimes

#### Pattern 1
| *Feature* | *Notable Values* |
| --- | --- |
| 12 to 17 | 182 |
| Male | 171 |
| White | 91 |
| Black | 86 |

**Interpretation:**
- High youth involvement (12–17)
- High male counts
- Roughly even Black/White representation
- Suggests youth-driven crime, possibly school/community level

#### Pattern 2
| *Feature* | *Notable Values* |
| --- | --- |
| 12 to 17 | 414 |
| Male | 399 |
| White | 258 |
| Black | 129 |

**Interpretation:**
- Extremely high volume
- Majority male youth crimes
- Strongly White-dominated
- Could reflect rural or suburban regions with high reported youth crime

#### Pattern 3
| *Feature* | *Notable Values* |
| --- | --- |
| Male | 45 |
| Black | 34 |
| 12 to 17 | 48 |

**Interpretation:**
- Mid-level youth crime
- Higher representation of Black offenders
- Possibly urban youth crime clusters

#### Pattern 4
| *Feature* | *Notable Values* |
| --- | --- |
| Male | 82 |
| White | 22 |
| Black | 62 |
| 12 to 17 | 94 |

**Interpretation:**
- Lower overall volume
- Includes notable Native American representation
- Could indicate patterns from tribal or rural areas with different reporting styles or issues

#### Assign patterns to reflect a demographically distinct crime profile
- Pattern 0 --> Low-volume youth crime
- Pattern 1 --> High Volume mixed-race youth crime
- Pattern 2 --> Suburban White-dominated high youth crime
- Patttern 3 --> Urban Black youth crime
- Pattern 4 --> Mid to High Volume Black youth crime


#### Transition Modeling - How Patterns Evolve Over Time
Build a transition matrix that shows how states move from one pattern to another over the years.

In [12]:
transitions = defaultdict(lambda: defaultdict(int))

# loop over each state to collect pattern transitions year over year
for state in model_df['State'].unique():
    state_data = model_df[model_df['State'] == state].sort_values('Year')
    patterns = state_data['Pattern'].tolist()

    for i in range(len(patterns) - 1):
        from_pattern = patterns[i]
        to_pattern = patterns[i+1]
        transitions[from_pattern][to_pattern] += 1

# covert to a probability matrix
# normalize transition counts to probabilities 
transition_matrix = {
    from_p: {
        to_p: count / sum(to_counts.values())
        for to_p, count in to_counts.items()
    }
    for from_p, to_counts in transitions.items()
}

# create a full matrix of all posivle transitions
patterns = sorted(model_df['Pattern'].unique())
matrix_df = pd.DataFrame(index=patterns, columns=patterns).fillna(0.0)

for from_p, to_dict in transition_matrix.items():
    for to_p, prob in to_dict.items():
        matrix_df.loc[from_p, to_p] = prob

matrix_df.index.name = 'From'
matrix_df.columns.name = 'To'

matrix_df.round(3)


To,0,1,2,3,4
From,,,,,
0,0.914,0.000,0.000,0.085,0.001
1,0.000,0.828,0.016,0.141,0.016
2,0.000,0.125,0.875,0.000,0.000
3,0.285,0.015,0.000,0.692,0.008
4,0.000,0.286,0.000,0.571,0.143


#### Analysis of Transitio Probability Matrix

#### Pattern 0
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 0 | 91% |
| Moves to Pattern 3  | 8% |

**Interpretation:**
- Highly stable pattern — most states stay here.
- Small chance of transitioning to Patterns 3 or 4.
- Likely represents a baseline or low-risk demographic crime pattern.

#### Pattern 1
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 1 | 83% |
| Moves to Pattern 2  | 2% |
| Moves to Pattern 3 | 14% |

**Interpretation:**
- Also fairly stable.
- But interestingly, 14% of transitions go to Pattern 3, possibly suggesting escalation.
- Could represent youth-heavy or racially mixed crime patterns shifting toward higher intensity.

#### Pattern 2
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 2 | 88% |
| Moves to Pattern 1  | 13% |

**Interpretation:**
- Somewhat stable, but with a substantial chance of returning to Pattern 1.
- Suggests an oscillating behavior between two patterns — possibly linked to policy shifts or local enforcement.

#### Pattern 3
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 3 | 69% |
| Moves to Pattern 0  | 29% |

**Interpretation:**
- Pattern 3 is less stable, often falling back to Pattern 0.
- May represent transient crime types that resolve or deescalate.

#### Pattern 4
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 4 | 14% |
| Moves to Pattern 3  | 57% |
| Moves to Pattern 1 | 29% |

**Interpretation:**
- Very unstable pattern — 86% of the time it transitions away.
- Most often into Pattern 3, possibly indicating dissipation of crime risk.
- May represent special, high-intensity or isolated crime patterns.

#### Overal Insights on the Pattern Transitions
- Patterns 0 and 1 are “attractors” — states tend to settle into them.
- Pattern 4 is volatile and quickly disperses — could be a high-risk but temporary scenario.
- Frequent returns to Pattern 0 suggest it might be a “default” or low-risk state.

### Risk Scoring & Forecasting
Assign a risk score to each predicted pattern for each state in a future year, then visualize the risk across the U.S.

In [14]:
pattern_risks = {
    0: 0.2, # low volume, mixed demographics
    1: 0.7, # high volume, Black/White balances
    2: 0.9, # extreme volume, White dominated
    3: 0.4, # mid volume, Black dominated
    4: 0.5, # mid to high volume but unstable 
}

def forecast_pattern(state, start_year, num_years, df, matrix):
    forecasted_patterns = {}
    
    # get initial patterns from the last real year
    row = df[(df['State'] == state) & (df['Year'] == start_year)]
    if row.empty:
        return forecasted_patterns
    
    current_pattern = row['Pattern'].values[0]
    
    for i in range(1, num_years + 1):
        next_year = start_year + i
        transitions = matrix.get(current_pattern, {})
        if transitions:
            next_pattern = max(transitions.items(), key=lambda x: x[1])[0]
        else:
            next_pattern = current_pattern
        
        forecasted_patterns[next_year] = next_pattern
        current_pattern = next_pattern
    
    return forecasted_patterns

# build transition matrix from historical data
transition_matrix = defaultdict(lambda: defaultdict(int))

for state in model_df['State'].unique():
    state_data = model_df[model_df['State'] == state].sort_values('Year')
    for i in range(len(state_data) - 1):
        current = state_data.iloc[i]['Pattern']
        next_ = state_data.iloc[i + 1]['Pattern']
        transition_matrix[current][next_] += 1

# Normalize transition counts to probabilities
for from_pattern in transition_matrix:
    total = sum(transition_matrix[from_pattern].values())
    for to_pattern in transition_matrix[from_pattern]:
        transition_matrix[from_pattern][to_pattern] /= total


start_year = 2017
num_years = 3
multi_year_forecast = []

for state in model_df['State'].unique():
    pattern_sequence = forecast_pattern(
        state,
        start_year=start_year,
        num_years=num_years,
        df=model_df,
        matrix=transition_matrix
    )

    for year, pattern in pattern_sequence.items():
        risk = pattern_risks.get(pattern, 0.0)
        multi_year_forecast.append({
            'State': state,
            'Year': year,
            'Predicted Pattern': pattern,
            'Risk Score': risk
        })

multi_forecast_df = pd.DataFrame(multi_year_forecast)
multi_forecast_df


,State,Year,Predicted Pattern,Risk Score
0,AK,2018,0,0.2
1,AK,2019,0,0.2
2,AK,2020,0,0.2
3,AR,2018,0,0.2
4,AR,2019,0,0.2
...,...,...,...,...
142,WV,2019,0,0.2
143,WV,2020,0,0.2
144,WY,2018,0,0.2
145,WY,2019,0,0.2


In [15]:
fig = px.choropleth(
    multi_forecast_df,
    locations='State',
    locationmode='USA-states',
    color='Risk Score',
    scope='usa',
    animation_frame='Year',
    color_continuous_scale='Reds',
    range_color=(0, 1),
    labels={'Risk Score': 'Crime Risk'},
    title="Forecasted Crime Risk by State (Animated Over Years)"
)

fig.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    margin=dict(l=20, r=20, t=50, b=20),
    coloraxis_colorbar=dict(
        title="Risk Score",
        ticks="outside"
    )
)

fig.show()

### Evaluate the Model: Compare Clusters with Preducated Pattern

In [ ]:
# Load dataset
full_df = pd.read_csv("dataset_1980_2020.csv")
full_df = full_df.rename(columns={'Count': 'Year'})

# Filter for evaluation years
eval_df = full_df[full_df['Year'].between(2018, 2020)]

# Aggregate and normalize
agg_eval = eval_df.groupby(['State', 'Year'])[offender_columns].sum().reset_index()
features_eval = scaler.transform(agg_eval[offender_columns])
agg_eval['Actual Pattern'] = kmeans.predict(features_eval)

# Merge forecasted and actual patterns
comparison_df = pd.merge(
    multi_forecast_df,
    agg_eval[['State', 'Year', 'Actual Pattern']],
    on=['State', 'Year'],
    how='inner'
)

from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix

y_true = comparison_df['Actual Pattern']
y_pred = comparison_df['Predicted Pattern']

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Adjusted Rand Index:", adjusted_rand_score(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


Accuracy: 0.8424657534246576
Adjusted Rand Index: 0.4647830984497608
Confusion Matrix:
 [[100   0  11   0]
 [  0   0   1   0]
 [ 10   0  23   0]
 [  0   0   1   0]]
